# ColonoMind TMC-UCM Ensemble Voting Evaluation
This notebook evaluates an ensemble of 5 Hybrid Models trained on TMC-UCM (Patient-Level Split).
It evaluates them on completely unseen domains: NTUH and LIMUC.

**Voting Logic:**
- All 5 models predict an image (using internal 0.70 Confidence Hybrid Routing).
- The ensemble takes a Majority Vote with thresholds of 3/5, 4/5, or 5/5.
- If agreement is below threshold, the image is flagged: **Refer to Doctor**.



In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import cv2
import pywt
import scipy.stats
from skimage.feature import graycomatrix, graycoprops
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from collections import Counter
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt



## 1. Setup Data Paths and Config


In [ ]:
BASE_DIR = '/home/D13K48009/raid/Clara/new_drive'
MODELS_DIR = f'{BASE_DIR}/Result/Intra_TMC-UCM'
TEST_DATASETS = {
    'NTUH': [f'{BASE_DIR}/Dataset+Code/MES classification_20250313', f'{BASE_DIR}/Dataset+Code/MES classification_20250724'],
    'LIMUC': [f'{BASE_DIR}/Dataset/LIMUC/train_and_validation_sets', f'{BASE_DIR}/Dataset/LIMUC/test_set']
}
MODEL_NAMES = ['ResNet-50', 'DenseNet-121', 'EfficientNet-B4', 'ConvNeXt-Tiny', 'ViT-B-16']
CLASS_NAMES = ['MES0', 'MES1', 'MES2', 'MES3']
DATASET_CLASS_FOLDERS = {
    'NTUH': ['MES0', 'MES1', 'MES2', 'MES3'],
    'LIMUC': ['Mayo 0', 'Mayo 1', 'Mayo 2', 'Mayo 3']
}
FOLDER_TO_LABEL = {
    'MES0': 'MES0', 'MES1': 'MES1', 'MES2': 'MES2', 'MES3': 'MES3',
    'Mayo 0': 'MES0', 'Mayo 1': 'MES1', 'Mayo 2': 'MES2', 'Mayo 3': 'MES3'
}
THRESHOLD_CONFIDENCE = 0.70



## 2. Helper Functions (Loading Data & Features)


In [ ]:
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        pt_1 = tf.where(tf.equal(y_true, 1), y_pred, tf.ones_like(y_pred))
        pt_0 = tf.where(tf.equal(y_true, 0), y_pred, tf.zeros_like(y_pred))
        return -tf.reduce_sum(alpha * tf.pow(1. - pt_1, gamma) * tf.math.log(pt_1 + 1e-7)) \
               -tf.reduce_sum((1 - alpha) * tf.pow(pt_0, gamma) * tf.math.log(1. - pt_0 + 1e-7))
    return focal_loss_fixed

def extract_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    coeffs2 = pywt.dwt2(gray, 'db1')
    LL, (LH, HL, HH) = coeffs2
    def stats(subband):
        return [np.mean(subband), np.std(subband), np.var(subband), scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6)]
    hh_energy = np.sum(np.square(HH)) / HH.size
    wavelet = stats(LL) + stats(LH) + stats(HL) + stats(HH) + [hh_energy]

    distances = [1, 3, 5]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(gray, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
    glcm_feat = [
        np.mean(graycoprops(glcm, 'contrast')),
        np.mean(graycoprops(glcm, 'dissimilarity')),
        np.mean(graycoprops(glcm, 'homogeneity'))
    ]
    return wavelet + glcm_feat

def load_dataset(dataset_name, paths):
    all_imgs, all_feats, all_labels = [], [], []
    folder_names = DATASET_CLASS_FOLDERS.get(dataset_name, CLASS_NAMES)
    for p in paths:
        for folder in folder_names:
            cls_dir = os.path.join(p, folder)
            if not os.path.exists(cls_dir): continue
            for file_name in os.listdir(cls_dir):
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(cls_dir, file_name)
                    img = cv2.imread(img_path)
                    if img is not None:
                        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img_resized = cv2.resize(img_rgb, (224, 224))
                        all_imgs.append(img_resized)
                        all_feats.append(extract_features(img_resized))
                        all_labels.append(FOLDER_TO_LABEL[folder])
    return np.array(all_imgs, dtype=np.float32), np.array(all_feats), np.array(all_labels)



## 3. Load 5 Models Trained on TMC-UCM


In [ ]:
models, scalers, agents = {}, {}, {}
for model_name in MODEL_NAMES:
    exp_dir = f'{MODELS_DIR}/{model_name}_Experiment'
    if not os.path.exists(exp_dir):
        print(f'❌ Missing model {model_name}')
        continue
    keras_model = load_model(f'{exp_dir}/{model_name}_hybrid.h5', custom_objects={'focal_loss_fixed': focal_loss(gamma=2.5, alpha=0.25)})
    scaler_ag = joblib.load(f'{exp_dir}/{model_name}_scaler.pkl')
    agent_model = lgb.Booster(model_file=f'{exp_dir}/{model_name}_agent.txt')
    models[model_name] = keras_model
    scalers[model_name] = scaler_ag
    agents[model_name] = agent_model
    print(f'✅ Loaded {model_name}')



## 4. Run Voting Ensemble Evaluation


In [ ]:
def get_majority_vote(predictions, threshold):
    counter = Counter(predictions)
    most_common_pred, count = counter.most_common(1)[0]
    return most_common_pred if count >= threshold else -1

le = LabelEncoder()
le.fit(CLASS_NAMES)

for test_dataset, paths in TEST_DATASETS.items():
    print(f'\n=============================================')
    print(f'🧪 Testing Ensemble on {test_dataset}')
    print(f'=============================================')
    X_img, X_feat, labels = load_dataset(test_dataset, paths)
    y_true = le.transform(labels)
    
    # Using ResNet-50's UMAP & Scaler as the global feature projection (as trained on TMC-UCM)
    exp_dir = f'{MODELS_DIR}/ResNet-50_Experiment'
    feat_scaler = joblib.load(f'{exp_dir}/scaler.pkl')
    umap_reducer = joblib.load(f'{exp_dir}/umap_model.pkl')
    
    X_feat_sc = feat_scaler.transform(X_feat)
    X_umap = umap_reducer.transform(X_feat_sc)
    
    all_preds = []
    for name in MODEL_NAMES:
        y_proba = models[name].predict([X_img, X_feat_sc, X_umap], verbose=0)
        y_deep = np.argmax(y_proba, axis=1)
        conf_deep = np.max(y_proba, axis=1)
        
        df_ag = pd.DataFrame(X_feat_sc, columns=[f'f{j}' for j in range(20)])
        df_ag['confidence'] = conf_deep
        df_ag['umap_0'] = X_umap[:, 0]
        df_ag['umap_1'] = X_umap[:, 1]
        features = ['confidence', 'umap_0', 'umap_1'] + [f'f{j}' for j in range(20)]
        
        X_te_ag = scalers[name].transform(df_ag[features].values)
        y_ag_proba = agents[name].predict(X_te_ag)
        y_agent = np.argmax(y_ag_proba, axis=1)
        
        # Internal Hybrid Routing for this specific model
        y_hybrid = np.where(conf_deep < THRESHOLD_CONFIDENCE, y_agent, y_deep)
        all_preds.append(y_hybrid)
        
    all_preds = np.array(all_preds).T # Shape: (N_samples, 5 models)
    
    for v_thresh in [3, 4, 5]:
        final_preds = np.array([get_majority_vote(p, v_thresh) for p in all_preds])
        
        processed_mask = final_preds != -1
        referred = np.sum(~processed_mask)
        total = len(y_true)
        acc = accuracy_score(y_true[processed_mask], final_preds[processed_mask]) if np.sum(processed_mask) > 0 else 0.0
        
        print(f'\n--- Voting Threshold: {v_thresh}/5 ---')
        print(f'Total Images: {total}')
        print(f'Referred to Doctor: {referred} ({(referred/total)*100:.2f}%)')
        print(f'Framework Accuracy: {acc*100:.2f}%')
        
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            cls_mask = (y_true == cls_idx) & processed_mask
            cls_total = np.sum((y_true == cls_idx) & processed_mask)
            if cls_total > 0:
                c_acc = np.sum(final_preds[cls_mask] == y_true[cls_mask]) / cls_total
                print(f'  {cls_name} Accuracy: {c_acc*100:.2f}%')
            else:
                print(f'  {cls_name} Accuracy: N/A (all referred)')

